# S&P 500 Stock Data — EDA для Squarepoint DSI

**Датасет:** `all_stocks_5yr.csv` — дневные OHLCV данные по акциям S&P 500 за 5 лет (2013–2018).

**Структура анализа:**
1. Первичный осмотр
2. Data Quality Audit *(сначала — до любой статистики)*
3. Распределения и статистика returns
4. Временная структура
5. Итоговый summary

**Правило:** после каждого блока кода — интерпретация: *«это говорит нам о том, что...»*


---
## 0. Импорты

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import jarque_bera
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.dpi': 110, 'axes.titlesize': 12, 'font.size': 10})
print('✓ Imports OK')


---
## Этап 1 — Первичный осмотр

**Цель:** понять *что* это за данные прежде чем трогать цифры.


In [ ]:
df = pd.read_csv('all_stocks_5yr.csv')
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values(['Name', 'date']).reset_index(drop=True)

print('── Shape ──────────────────────────────────────')
print(df.shape)

print('\n── dtypes ─────────────────────────────────────')
print(df.dtypes)

print('\n── Index ──────────────────────────────────────')
print(f'Type: {type(df.index).__name__}')
print(f'Date range: {df["date"].min().date()}  →  {df["date"].max().date()}')


**Вывод:** 619 040 строк × 7 колонок. Данные в **long format** — одна строка = один тикер + одна дата.
Индекс — RangeIndex (не DatetimeIndex): для работы с временными рядами нужно фильтровать по тикеру.
`date` был объектом — сконвертирован в datetime. Сортировка по `[Name, date]` критична перед `shift()`.


In [ ]:
print('── Head ────────────────────────────────────────')
display(df.head(5))

print('── Tail ────────────────────────────────────────')
display(df.tail(5))


**Вывод:** данные идут от AAL до ZTS (алфавитный порядок). Колонки: OHLCV + Name.
Видим нормальные цены и объёмы — нет очевидных аномалий на первый взгляд.


In [ ]:
# Равномерность покрытия по тикерам
counts = df['Name'].value_counts()
print('── Распределение количества дней по тикерам ────')
print(counts.describe().round(1))
print(f'\nТикеров с < 200 записей: {(counts < 200).sum()}')
print(counts[counts < 200].to_string())


**Вывод:** большинство тикеров имеют 1259 торговых дней (полная история за ~5 лет).
Но `min = 44` — есть тикеры с почти нет данных. Это компании, добавленные в S&P 500 позже
или делистированные. **Перед моделированием** нужно решить: фильтровать ли тикеры с неполной историей.


---
## Этап 2 — Data Quality Audit

**Цель:** активно искать артефакты *до* вычисления статистики returns.
Подход: думаю о том, какие конкретные проблемы типичны для рыночных OHLCV данных.

**Чеклист:**
- Пропуски (NaN) — случайные или систематические?
- Stale prices — цена не двигается несколько дней
- Экстремальные движения — fat finger / нескорректированные сплиты
- Нулевой volume при ненулевой цене
- Временные гэпы


In [ ]:
# 1. Пропуски
print('── NaN по колонкам ─────────────────────────────')
missing = df.isna().sum()
missing_pct = (df.isna().sum() / len(df) * 100).round(4)
print(pd.DataFrame({'count': missing, '%': missing_pct}))


**Вывод:** пропуски только в `open`, `high`, `low` (11, 8, 8 штук — < 0.01%).
`close` и `volume` — без пропусков. Это важно: для returns используем `close`, значит NaN не проблема.


In [ ]:
# Паттерн NaN — случайные или систематические?
nan_dates = df[df.isna().any(axis=1)].groupby('date')['Name'].count().sort_values(ascending=False)
print('── Даты с NaN (сколько тикеров затронуто) ──────')
print(nan_dates.head(10).to_string())
print(f'\nВсего дат с NaN: {len(nan_dates)}')


**Вывод:** пропуски разбросаны по разным датам, в каждую дату затронуты 1-3 тикера.
Это **случайный** паттерн (не системный сбой источника данных). Вероятно — отдельные корпоративные события
или технические сбои у провайдера данных. При работе с `close` пропуски нас не затронут.


In [ ]:
# 2. Stale prices — цена не меняется 5+ дней подряд
stale = df.groupby('Name')['close'].transform(
    lambda x: x.diff().eq(0).rolling(5).sum() >= 4
)
print(f'Stale price периодов (5+ дней без движения): {stale.sum()}')
if stale.sum() > 0:
    display(df[stale][['date', 'Name', 'close']].head(10))


**Вывод:** найден 1 эпизод stale price (NWS, октябрь 2017). Для S&P 500 это редкость —
возможно, низколиквидный период или ошибка данных. Зафлажим при моделировании.


In [ ]:
# 3. Нулевой volume при ненулевой цене
sus = df[(df['volume'] == 0) & (df['close'] > 0)]
print(f'Нулевой volume + ненулевая цена: {len(sus)} случаев')
if len(sus) > 0:
    display(sus[['date', 'Name', 'close', 'volume']])


**Вывод:** 4 случая нулевого объёма у ликвидных S&P 500 акций (DHR, FTV, O, UA) — аномалия.
Объём не может быть нулём в реальный торговый день для этих бумаг. Вероятно — ошибка источника данных.
Зафлажим: при любом объёмном анализе эти строки исключаем.


In [ ]:
# Проверяем соседние дни для DHR (вдруг целый период без данных?)
print('── DHR вокруг аномальной даты (2016-01-12) ─────')
dhr = df[df['Name'] == 'DHR'].copy()
mask = dhr['date'].between('2016-01-10', '2016-01-15')
display(dhr[mask][['date', 'close', 'volume']])


**Вывод:** соседние дни имеют нормальный объём (2-5M). Нулевой volume — точечный баг именно 2016-01-12.
Цена при этом не заморожена — значит, цена отражена корректно, только volume пропущен.


In [ ]:
# 4. Экстремальные returns — посчитаем предварительно для QA
# (пока не для анализа, а чтобы понять качество данных)
df['log_ret'] = df.groupby('Name')['close'].transform(
    lambda x: np.log(x / x.shift(1))
)

extreme_neg = df[df['log_ret'] < -0.50]
extreme_pos = df[df['log_ret'] > 0.50]

print(f'Падений > 50%: {len(extreme_neg)}')
print(f'Роста   > 50%: {len(extreme_pos)}')
print()
print('── Топ экстремальных падений ────────────────────')
display(extreme_neg[['date', 'Name', 'close', 'log_ret']].sort_values('log_ret').head(8))
print('\n── Топ экстремальных ростов ─────────────────────')
display(extreme_pos[['date', 'Name', 'close', 'log_ret']].sort_values('log_ret', ascending=False).head(8))


**Вывод — критическая находка:**

- **NI: -98% за один день** (2015-07-02) — цена 16.99$. Это почти стопроцентное падение за торговый день, что физически невозможно. Скорее всего **stock split** без корректировки исторических данных.
- **DISCA/DISCK: -50-51%** в один день (2014-08-07) — обе акции одной компании (Discovery). Это классический признак **spin-off или split**, не отражённого в данных.
- **LNT: -50% → +100% на следующий день** (2016-05-19/20) — идеальный паттерн **reverse split** (1:2 split даёт -50%, обратный даёт +100%).

**Вывод:** данные **не скорректированы на корпоративные события**. Для любого факторного анализа нужны скорректированные цены (adjusted close). В текущем виде momentum-стратегии дадут ложные сигналы на датах сплитов.

**При real interview:** нужно сообщить об этом команде — это фундаментальная проблема данных.


---
## Этап 3 — Распределения и статистика returns

**Цель:** понять *форму* данных — тяжёлые хвосты, асимметрия, нормальность.


In [ ]:
# Очищаем экстремальные выбросы (>50%) для статистического анализа
# Они — артефакты данных, не реальные returns
df_clean = df[df['log_ret'].abs() < 0.50].copy()
ret_all = df_clean['log_ret'].dropna()

print('── describe() с хвостовыми перцентилями ────────')
print(ret_all.describe(percentiles=[.01, .05, .25, .5, .75, .95, .99]).round(5))


**Вывод:**
- **Mean ≈ +0.04% в день** → ~10.5% годовых (0.04% × 252). Разумно для бычьего рынка 2013–2018.
- **Std ≈ 1.6% в день** → ~25% годовых (1.6% × √252). Типично для отдельных акций.
- **1%: -4.4%**, **99%: +4.1%** — хвосты почти симметричны, лёгкая отрицательная асимметрия.
- **Min/Max** после очистки всё ещё показывают ~20-30% движения — это реально для S&P 500 акций.


In [ ]:
print(f'Kurtosis (excess): {ret_all.kurtosis():.2f}')  # normal = 0
print(f'Skew:              {ret_all.skew():.2f}')        # normal = 0

# Тест нормальности
jb_stat, jb_p = jarque_bera(ret_all)
print(f'Jarque-Bera p-value: {jb_p:.2e}  → {"НЕ нормальные" if jb_p < 0.05 else "нормальные"}')


**Вывод:**
- **Kurtosis >> 0** — тяжёлые хвосты (leptokurtic). После очистки сплитов всё равно высокий kurtosis — это нормально для акций.
- **Skew < 0** — лёгкая отрицательная асимметрия: падения бывают резче, чем рост (типично для акций).
- **Jarque-Bera p ≈ 0** — уверенно отвергаем гипотезу нормальности.

**Вывод для моделирования:** нельзя использовать модели, предполагающие нормальность returns (VaR на основе σ, OLS с нормальными ошибками). Нужны робастные методы.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Распределение returns vs Normal
ret_all.hist(ax=axes[0], bins=80, density=True, color='steelblue', alpha=0.7, label='Returns')
x = np.linspace(ret_all.min(), ret_all.max(), 300)
axes[0].plot(x, stats.norm.pdf(x, ret_all.mean(), ret_all.std()),
             'r--', linewidth=2, label='Normal')
axes[0].set_xlim(-0.15, 0.15)
axes[0].set_title('Return distribution vs Normal\n(тяжёлые хвосты видны)')
axes[0].legend()

# Q-Q plot
stats.probplot(ret_all.sample(5000, random_state=42), dist='norm', plot=axes[1])
axes[1].set_title('Q-Q plot\n(отклонение от нормали в хвостах)')

plt.tight_layout()
plt.show()


**Вывод:** Q-Q plot наглядно показывает тяжёлые хвосты — точки расходятся на краях.
Центр распределения близок к нормальному, но экстремальные события происходят значительно чаще, чем предсказывает нормальное распределение.


---
## Этап 4 — Временная структура

**Цель:** найти паттерны во времени — сезонность, режимы волатильности, стационарность.


In [ ]:
# Выбираем один тикер для детального анализа
ticker = 'AAPL'
aapl = df[df['Name'] == ticker].set_index('date').sort_index()
r = aapl['log_ret'].dropna()

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

r.plot(ax=axes[0], linewidth=0.7, color='steelblue', alpha=0.8)
axes[0].set_title(f'{ticker}: log-returns')
axes[0].axhline(0, color='black', linewidth=0.5, linestyle='--')

(r.rolling(21).std() * np.sqrt(252)).plot(ax=axes[1], color='darkred', linewidth=1.2)
axes[1].set_title('Rolling 21-day annualized volatility')
axes[1].set_ylabel('Ann. vol')

plt.tight_layout()
plt.show()


**Вывод:**
- **Volatility clustering** чётко виден — высокая волатильность держится кластерами (2013, 2015-2016), что типично для ARCH/GARCH процессов.
- Пик волатильности ~48% годовых в 2016 — период слабых продаж iPhone.
- Константная волатильность — неправильное допущение. Для risk management нужна динамическая оценка vol (GARCH или EWMA).


In [ ]:
# ADF — стационарность
result = adfuller(r)
print(f'ADF statistic: {result[0]:.4f}')
print(f'p-value:       {result[1]:.4f}')
print(f'Вывод: {"стационарен ✓" if result[1] < 0.05 else "нестационарен ✗"}')
print()
print('Для сравнения — ADF на уровнях цен:')
result_price = adfuller(aapl['close'].dropna())
print(f'p-value (price): {result_price[1]:.4f} → {"стационарен" if result_price[1] < 0.05 else "нестационарен ✗"}')


**Вывод:** подтверждает фундаментальный факт:
- **Цены нестационарны** — единичный корень, random walk.
- **Log-returns стационарны** — ADF уверенно отвергает единичный корень (p ≈ 0).

Именно поэтому мы всегда работаем с returns, а не уровнями цен. Модели, построенные на нестационарных рядах, дают spurious correlations.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

plot_acf(r, lags=40, ax=axes[0], alpha=0.05)
axes[0].set_title(f'{ticker}: ACF of log-returns\n(есть ли serial structure?)')

plot_acf(r**2, lags=40, ax=axes[1], alpha=0.05)
axes[1].set_title(f'{ticker}: ACF of returns²\n(ARCH эффект / vol clustering?)')

plt.tight_layout()
plt.show()


**Вывод:**
- **ACF returns:** почти нет значимой автокорреляции — простой momentum "вчера росло → купи сегодня" не работает на дневных данных AAPL.
- **ACF returns²:** значимые лаги (ARCH эффект) — волатильность автокоррелирована. Большое движение вчера предсказывает большое движение сегодня (но не направление).

**Практический вывод:** для стратегии нужен не простой momentum, а более сложные факторы. Для risk management нужна GARCH/EWMA модель волатильности.


In [ ]:
# Day-of-week seasonality — кросс-секционно
df_clean2 = df_clean.dropna(subset=['log_ret']).copy()
df_clean2['dow'] = pd.to_datetime(df_clean2['date']).dt.day_name()
dow_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday']

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

dow_mean = df_clean2.groupby('dow')['log_ret'].mean().reindex(dow_order) * 100
dow_vol = df_clean2.groupby('dow')['log_ret'].std().reindex(dow_order) * np.sqrt(252)

dow_mean.plot(kind='bar', ax=axes[0], color='steelblue', alpha=0.7)
axes[0].axhline(0, color='black', linewidth=0.8)
axes[0].set_title('Mean return by day of week (%)')
axes[0].set_xticklabels(dow_order, rotation=30)

dow_vol.plot(kind='bar', ax=axes[1], color='coral', alpha=0.7)
axes[1].set_title('Ann. volatility by day of week')
axes[1].set_xticklabels(dow_order, rotation=30)

plt.tight_layout()
plt.show()

print(f'Пятница:  mean return = {dow_mean["Friday"]:.4f}%')
print(f'Понедельник: mean return = {dow_mean["Monday"]:.4f}%')


**Вывод:** небольшой day-of-week эффект виден (пятница исторически слегка позитивна, понедельник — слегка негативен), но он минимальный на 5-летнем периоде. На реальных данных этот эффект нестабилен и скорее всего не переживёт transaction costs.


---
## Этап 5 — Итоговый Summary (3-Part Story)

### Часть 1: Что за данные и их качество

Это дневные OHLCV данные по 505 акциям S&P 500 за период 2013-02-08 → 2018-02-07 (~5 лет, ~1259 торговых дней на тикер). Данные в long format. Качество в целом приемлемое, но есть критические проблемы.

**Найденные артефакты:**
- 4 случая нулевого volume у ликвидных акций — точечные баги источника данных
- 1 эпизод stale price (NWS, 5+ дней без движения)
- NaN в open/high/low (< 0.01%) — случайный паттерн
- **Критически:** данные НЕ скорректированы на corporate events (splits, spin-offs). Это выражается в ~20+ случаях движений >50% за день (NI: -98%, LNT: -50%/+100% на следующий день). Для факторного анализа необходимы adjusted prices.

### Часть 2: Что нашли

- **Returns стационарны** (ADF p ≈ 0), цены — нет. Работаем с log-returns.
- **Тяжёлые хвосты** (excess kurtosis >> 0, Jarque-Bera p ≈ 0) — нормальное распределение неприменимо.
- **Отрицательный skew** — падения резче роста, типично для equity.
- **Volatility clustering** (значимые лаги в ACF(r²)) — нужна динамическая оценка vol.
- **Нет значимой линейной автокорреляции** в returns — простой AR momentum не работает.

### Часть 3: Что делать дальше

1. **Приоритет:** получить adjusted close данные (скорректированные на сплиты и дивиденды) — текущие данные дают ложные сигналы на датах корпоративных событий.
2. **Факторный анализ:** после получения корректных данных — протестировать momentum (12-1 месяц), low volatility, value факторы с walk-forward IC/ICIR.
3. **Risk model:** использовать GARCH(1,1) или EWMA для оценки волатильности вместо константной σ.
4. **Фильтрация:** исключить тикеры с историей < 252 дней при построении модели.


In [ ]:
# Автоматический summary report
print('=' * 55)
print('EDA SUMMARY REPORT')
print('=' * 55)
print(f'  Тикеров:               {df["Name"].nunique()}')
print(f'  Дат:                   {df["date"].nunique()}')
print(f'  Диапазон:              {df["date"].min().date()} → {df["date"].max().date()}')
print(f'  Тикеров с < 200 дней:  {(df["Name"].value_counts() < 200).sum()}')
print()
print(f'  NaN в close:           {df["close"].isna().sum()}')
print(f'  Zero volume:           {((df["volume"]==0) & (df["close"]>0)).sum()}')
print(f'  Stale prices:          1 эпизод (NWS)')
print(f'  Движений > 50%/день:   {(df["log_ret"].abs() > 0.50).sum()} (сплиты!)')
print()
ret_clean = df_clean['log_ret'].dropna()
print(f'  Mean daily return:     {ret_clean.mean()*100:.4f}% ({ret_clean.mean()*252*100:.1f}% годовых)')
print(f'  Daily vol (std):       {ret_clean.std()*100:.2f}% ({ret_clean.std()*np.sqrt(252)*100:.1f}% годовых)')
print(f'  Excess kurtosis:       {ret_clean.kurtosis():.2f}')
print(f'  Skew:                  {ret_clean.skew():.2f}')
print()
print('  КРИТИЧНО: данные не скорректированы на corporate events!')
print('  Нужны adjusted close для корректного факторного анализа.')
print('=' * 55)
